# Proses lokal Sentinel-2: ekstraksi piksel dan klasifikasi tutupan lahan

## Business understanding

**Tujuan:** menghasilkan dataset berlabel untuk model tutupan lahan Jawa Timur—Built-up, Farm land, Water body, Forest, dan Mangrove—sebagai dukungan awal pemantauan wilayah dan penentuan lokasi yang perlu diperiksa lebih lanjut. Model/peta bukan peta legal dan tetap perlu validasi lapangan, terutama untuk membedakan mangrove dengan hutan biasa.

**Alur sumber data:** Copernicus Browser hanya digunakan untuk mengunduh produk Sentinel-2 L2A. Ekstrak ZIP secara lokal, kemudian gunakan JP2 B02/B03/B04/B08 (10 m) dan B11 (20 m). Notebook/script lokal menyelaraskan B11 ke grid B02, membaca piksel dalam poligon GeoJSON, menghitung indeks, membagi per ID poligon, dan menghasilkan CSV. Tidak ada pengunduhan citra dari Copernicus/STAC saat ekstraksi.

**Spesifikasi:** setiap baris CSV mewakili satu piksel valid di dalam poligon, berisi `ID_Poligon`, `kelas`, `status_split`, B02/B03/B04/B08/B11, NDVI, NDWI, dan NDBI. Band tetap dalam DN integer Sentinel-2 L2A; indeks dihitung sebagai rasio. Pembagian 80/20 dilakukan berdasarkan poligon dan distratifikasi kelas, jadi piksel dari ID poligon yang sama tidak masuk ke train dan test sekaligus.

## 1. Persiapkan file lokal

Ekstrak produk ZIP hasil unduhan ke satu folder, misalnya `data/sentinel2/`. Script mencari `*_B02_10m.jp2`, `*_B03_10m.jp2`, `*_B04_10m.jp2`, `*_B08_10m.jp2`, dan `*_B11_20m.jp2` secara rekursif. Nama produk/granul harus cocok lintas band; pisahkan akuisisi berbeda ke folder masing-masing. Citra harus mencakup lokasi poligon dan sebaiknya dipilih dengan tutupan awan rendah. Hasil ditulis ke `data/csv/split_dataset/train_dataset.csv` dan `test_dataset.csv`.

In [22]:
from pathlib import Path
import subprocess
import sys

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "jawatimur.geojson").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Jalankan notebook dari dalam repository PSD-B.")

geojson_path = repo_root / "data" / "jawatimur.geojson"
jp2_folder = repo_root / "data" / "sentinel2"  # Ganti jika ZIP diekstrak ke lokasi lain.
output_folder = repo_root / "data" / "csv" / "split_dataset"
extractor = repo_root / "src" / "1-project" / "extract_sentinel_pixels.py"

output_folder.mkdir(parents=True, exist_ok=True)

if not geojson_path.is_file():
    raise FileNotFoundError(f"GeoJSON tidak ditemukan: {geojson_path}")
if not jp2_folder.is_dir():
    raise FileNotFoundError(
        f"Direktori JP2 tidak ditemukan: {jp2_folder}. "
        "Ekstrak produk Sentinel-2 ke folder data/sentinel2 terlebih dahulu."
    )
if not extractor.is_file():
    raise FileNotFoundError(f"Script extractor tidak ditemukan: {extractor}")

result = subprocess.run(
    [
        sys.executable,
        str(extractor),
        "--geojson", str(geojson_path),
        "--band-dir", str(jp2_folder),
        "--out-dir", str(output_folder),
        "--test-size", "0.2",
        "--random-state", "42",
    ],
    cwd=str(repo_root),
    check=False,
    capture_output=True,
    text=True,
)

if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)

if result.returncode != 0:
    error_text = result.stderr or result.stdout or ""
    if "Application Control policy has blocked" in error_text:
        raise RuntimeError(
            "Windows memblokir DLL Rasterio dengan kebijakan Application Control. "
            "Ini bukan kesalahan GeoJSON atau baris returncode. Jalankan ekstraksi "
            "di lingkungan Python yang diizinkan (misalnya WSL/Linux yang disetujui) "
            "atau minta administrator mengizinkan Rasterio. Jangan menonaktifkan "
            "kebijakan keamanan."
        )
    if "No module named 'geopandas'" in error_text:
        raise RuntimeError(
            "GeoPandas belum terpasang pada interpreter notebook ini. Aktifkan "
            "environment proyek yang benar, lalu pasang dependensi proyek."
        )
    raise RuntimeError(
        f"Ekstraksi gagal (exit code {result.returncode}). Periksa pesan error "
        "yang tercetak di atas."
    )

train_csv = output_folder / "train_dataset.csv"
test_csv = output_folder / "test_dataset.csv"
print("Train:", train_csv)
print("Test:", test_csv)

Traceback (most recent call last):
  File "d:\PSD-B-main\src\1-project\extract_sentinel_pixels.py", line 18, in <module>
    import rasterio
  File "d:\PSD-B-main\.venv\Lib\site-packages\rasterio\__init__.py", line 37, in <module>
    from rasterio._base import DatasetBase
ImportError: DLL load failed while importing _base: An Application Control policy has blocked this file.



RuntimeError: Windows memblokir DLL Rasterio dengan kebijakan Application Control. Ini bukan kesalahan GeoJSON atau baris returncode. Jalankan ekstraksi di lingkungan Python yang diizinkan (misalnya WSL/Linux yang disetujui) atau minta administrator mengizinkan Rasterio. Jangan menonaktifkan kebijakan keamanan.

## 2. Periksa hasil ekstraksi dan kebocoran split

Jika error menyebut kelas/poligon tidak memiliki piksel, periksa apakah citra JP2 memang menutupi lokasi tersebut. Wilayah Jawa Timur dapat membutuhkan beberapa granula/produk Sentinel-2. Fitur menyimpan semua piksel yang valid di dalam poligon, sehingga CSV dapat berukuran besar.

In [ ]:
import pandas as pd

feature_columns = ["B02", "B03", "B04", "B08", "B11", "NDVI", "NDWI", "NDBI"]
train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)
required = {"ID_Poligon", "kelas", "status_split", *feature_columns}
for name, frame, expected_split in (
    ("train", train_df, "train"),
    ("test", test_df, "test"),
):
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{name} kehilangan kolom {sorted(missing)}")
    if set(frame["status_split"].unique()) != {expected_split}:
        raise ValueError(f"status_split salah pada {name}")
    if frame[feature_columns].isna().any().any():
        raise ValueError(f"Ada nilai fitur kosong pada {name}")

overlap = set(train_df["ID_Poligon"].astype(str)) & set(test_df["ID_Poligon"].astype(str))
if overlap:
    raise ValueError(f"ID poligon bocor train/test: {sorted(overlap)[:10]}")

display(pd.crosstab(train_df["kelas"], train_df["status_split"]).rename(columns={"train": "train pixels"}))
display(pd.crosstab(test_df["kelas"], test_df["status_split"]).rename(columns={"test": "test pixels"}))
print(f"Jumlah piksel train={len(train_df):,}; test={len(test_df):,}; ID overlap={len(overlap)}")
display(train_df.head())

## 3. Latih, evaluasi, dan lihat feature importance

Evaluasi dihitung pada poligon test yang terpisah dari poligon train. Macro-F1 dan skor per kelas lebih informatif untuk lima kelas daripada akurasi keseluruhan saja. Jika Mangrove dan Forest sering tertukar, tambah label mangrove/hutan yang representatif, cek tanggal/citra, dan lakukan validasi lapangan.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

class_names = ["Built-up", "Farm land", "Water body", "Forest", "Mangrove"]
for name, frame in (("train", train_df), ("test", test_df)):
    missing_classes = set(class_names) - set(frame["kelas"])
    if missing_classes:
        raise ValueError(f"Kelas tidak ada pada {name}: {sorted(missing_classes)}")

model = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)
model.fit(train_df[feature_columns], train_df["kelas"])
prediction = model.predict(test_df[feature_columns])
print(f"Macro-F1 test: {f1_score(test_df['kelas'], prediction, labels=class_names, average='macro'):.3f}")
print(classification_report(test_df["kelas"], prediction, labels=class_names, zero_division=0))
display(pd.DataFrame(
    confusion_matrix(test_df["kelas"], prediction, labels=class_names),
    index=[f"Aktual: {name}" for name in class_names],
    columns=[f"Prediksi: {name}" for name in class_names],
))

importance = pd.DataFrame({
    "fitur": feature_columns,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False, ignore_index=True)
display(importance)